# Multi-Asset Quantitative Market Risk & Stress Testing Framework

## Project Objective

This project develops a quantitative market-risk framework for a diversified multi-asset portfolio.

The main objectives are to:

- Collect and preprocess financial market data.
- Construct an equal-weighted multi-asset portfolio.
- Measure portfolio volatility and dependence between assets.
- Estimate Value at Risk (VaR) using Historical, Parametric and Monte Carlo methods.
- Estimate Expected Shortfall (CVaR).
- Model time-varying volatility using a GARCH(1,1) model.
- Backtest VaR forecasts using Kupiec and Christoffersen tests.
- Evaluate portfolio losses during historical and hypothetical stress scenarios.
- Compare the strengths and limitations of different market-risk models.

### Assets

- SPY — Equity
- EURUSD=X — Foreign Exchange
- TLT — Long-duration US Treasury exposure

The portfolio uses equal weights of one-third for each asset. The purpose is methodological risk analysis rather than investment optimization.

In [1]:
import pandas as pd
import numpy as np
import scipy
import matplotlib
import yfinance as yf
import arch

print("pandas:", pd.__version__)
print("numpy:", np.__version__)
print("scipy:", scipy.__version__)
print("matplotlib:", matplotlib.__version__)
print("yfinance:", yf.__version__)
print("arch:", arch.__version__)

pandas: 2.2.3
numpy: 2.1.3
scipy: 1.15.3
matplotlib: 3.10.0
yfinance: 1.7.0
arch: 8.0.0


In [2]:
import sys
from pathlib import Path

PROJECT_ROOT = Path.cwd().parent

if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

import pandas as pd
import numpy as np

pd.set_option("display.max_columns", None)
pd.set_option("display.float_format", lambda x: f"{x:.4f}")

from src.data.downloader import download_raw_data
from src.data.cleaner import clean_prices
from src.data.dataset import build_returns

print("Project root:", PROJECT_ROOT)
print("Imports successful.")

Project root: c:\Users\swaya\OneDrive\Documents\Multi-Asset-Market-Risk
Imports successful.


In [3]:
raw_data = download_raw_data(
    start="2005-01-01",
    overwrite=False
)

print("Downloaded/loaded:", list(raw_data.keys()))

[download] SPY -> C:\Users\swaya\OneDrive\Documents\Multi-Asset-Market-Risk\data\raw\SPY.csv
[download] EURUSD=X -> C:\Users\swaya\OneDrive\Documents\Multi-Asset-Market-Risk\data\raw\EURUSD.csv
[download] TLT -> C:\Users\swaya\OneDrive\Documents\Multi-Asset-Market-Risk\data\raw\TLT.csv
Downloaded/loaded: ['SPY', 'EURUSD', 'TLT']


In [4]:
prices = clean_prices()

print(prices.head())
print(prices.shape)
print(prices.isna().sum())

[saved] clean prices -> C:\Users\swaya\OneDrive\Documents\Multi-Asset-Market-Risk\data\processed\prices.csv
               SPY  EURUSD     TLT
Date                              
2005-01-03 81.1746  1.3470 43.7155
2005-01-04 80.1827  1.3282 43.2574
2005-01-05 79.6294  1.3280 43.4889
2005-01-06 80.0342  1.3183 43.5185
2005-01-07 79.9195  1.3061 43.6170
(5641, 3)
SPY       0
EURUSD    0
TLT       0
dtype: int64


In [5]:
returns = build_returns()

print(returns.head())
print(returns.describe())

[saved] daily log returns -> C:\Users\swaya\OneDrive\Documents\Multi-Asset-Market-Risk\data\processed\returns.csv
               SPY  EURUSD     TLT
Date                              
2005-01-04 -0.0123 -0.0141 -0.0105
2005-01-05 -0.0069 -0.0001  0.0053
2005-01-06  0.0051 -0.0073  0.0007
2005-01-07 -0.0014 -0.0093  0.0023
2005-01-10  0.0047  0.0037  0.0016
            SPY    EURUSD       TLT
count 5640.0000 5640.0000 5640.0000
mean     0.0004   -0.0000    0.0001
std      0.0117    0.0069    0.0090
min     -0.1159   -0.1433   -0.0690
25%     -0.0037   -0.0031   -0.0050
50%      0.0005   -0.0000    0.0000
75%      0.0056    0.0031    0.0053
max      0.1356    0.1596    0.0725


In [6]:
expected_assets = {"SPY", "EURUSD", "TLT"}

assert expected_assets.issubset(set(prices.columns)), \
    "Missing expected asset column(s) in prices."

assert expected_assets.issubset(set(returns.columns)), \
    "Missing expected asset column(s) in returns."

assert prices.isna().sum().sum() == 0, \
    "prices still contains missing values."

assert returns.isna().sum().sum() == 0, \
    "returns still contains missing values."

print("All validation checks passed.")

raw_dir = PROJECT_ROOT / "data" / "raw"

for name in ["SPY.csv", "EURUSD.csv", "TLT.csv"]:
    path = raw_dir / name
    print(f"{path} exists: {path.exists()}")

processed_dir = PROJECT_ROOT / "data" / "processed"

for name in ["prices.csv", "returns.csv"]:
    path = processed_dir / name
    print(f"{path} exists: {path.exists()}")

All validation checks passed.
c:\Users\swaya\OneDrive\Documents\Multi-Asset-Market-Risk\data\raw\SPY.csv exists: True
c:\Users\swaya\OneDrive\Documents\Multi-Asset-Market-Risk\data\raw\EURUSD.csv exists: True
c:\Users\swaya\OneDrive\Documents\Multi-Asset-Market-Risk\data\raw\TLT.csv exists: True
c:\Users\swaya\OneDrive\Documents\Multi-Asset-Market-Risk\data\processed\prices.csv exists: True
c:\Users\swaya\OneDrive\Documents\Multi-Asset-Market-Risk\data\processed\returns.csv exists: True
